kernel : Python (Pixi)

#### Main Dataset

In [23]:
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

series_name = "macnair"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")

macnair_adata = sc.read_h5ad(os.path.join(clustered_h5ad_dir, series_name + ".h5ad"))

In [4]:
plot.plot_umap(macnair_adata, series_name, has_celltype=True)

In [ ]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "CLDN11",  
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX5", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "SLC2A5", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(macnair_adata, series_name, cell_marker_genes, group = "leiden", project="ANNOTATION")

In [24]:
leiden_to_celltype = {
    "0": "Doublet",
    "1": "Astrocyte",
    "2": "InhibitoryNeuron",
    "3": "ExcitatoryNeuron",
    "4": "ExcitatoryNeuron",
    "5": "ExcitatoryNeuron",
    "6": "ExcitatoryNeuron",
    "7": "ExcitatoryNeuron",
    "8": "Oligodendrocyte",
    "9": "InhibitoryNeuron",
    "10": "OPC",
    "11": "Oligodendrocyte",
    "12": "Oligodendrocyte",
    "13": "ExcitatoryNeuron",
    "14": "ExcitatoryNeuron",
    "15": "InhibitoryNeuron",
    "16": "ExcitatoryNeuron",
    "17": "ExcitatoryNeuron",
    "18": "ExcitatoryNeuron",
    "19": "ExcitatoryNeuron",
    "20": "Oligodendrocyte",
    "21": "ExcitatoryNeuron",
    "22": "ExcitatoryNeuron",
    "23": "ExcitatoryNeuron",
    "24": "ExcitatoryNeuron",
    "25": "ExcitatoryNeuron",
    "26": "Astrocyte",
    "27": "Oligodendrocyte",
    "28": "ExcitatoryNeuron",
    "29": "InhibitoryNeuron",
    "30": "ExcitatoryNeuron",
    "31": "Astrocyte",
    "32": "InhibitoryNeuron",
    "33": "Bcell",
    "34": "Astrocyte",
    "35": "Oligodendrocyte",
    "36": "Doublet",
    "37": "InhibitoryNeuron",
    "38": "ExcitatoryNeuron",
    "39": "Astrocyte",
    "40": "Oligodendrocyte",
    "41": "Oligodendrocyte",
    "42": "Oligodendrocyte",
    "43": "Oligodendrocyte",
    "44": "Microglia",
    "45": "ExcitatoryNeuron",
    "46": "Oligodendrocyte",
    "47": "VLMC",
    "48": "Oligodendrocyte",
    "49": "Astrocyte",
    "50": "Pericyte",
    "51": "Astrocyte",
    "52": "Endothelial",
    "53": "Oligodendrocyte",
    "54": "ExcitatoryNeuron",
    "55": "Tcell",
    "56": "Oligodendrocyte",
    "57": "Oligodendrocyte",
    "58": "Microglia",
    "59": "InhibitoryNeuron",
    "60": "OPC",
    "61": "Oligodendrocyte",
    "62": "Microglia",
    "63": "Microglia", 
    "64": "Microglia",
    "65": "Microglia",
    "66": "Astrocyte",
    "67": "Ependymal",
    "68": "COP",
    "69": "Endothelial",
}

leiden_str = macnair_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_adata.obs["cell"] = mapped

macnair_adata = macnair_adata[macnair_adata.obs["cell"] != "Doublet"].copy()

In [16]:
macnair_adata.write_h5ad(os.path.join(final_h5ad_dir, series_name + "_final.h5ad"))

In [ ]:
pre_h5ad_dir = find_env_dir("PRE_H5AD")

opc = macnair_adata[
    (macnair_adata.obs["cell"] == "OPC") |
    (macnair_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(pre_h5ad_dir, series_name + "_opc_raw.h5ad"))

oligodendrocyte = macnair_adata[macnair_adata.obs["cell"] == "Oligodendrocyte"]
oligodendrocyte.write_h5ad(os.path.join(pre_h5ad_dir, series_name + "_oligodendrocyte_raw.h5ad"))

oligo = macnair_adata[
    (macnair_adata.obs["cell"] == "OPC") |
    (macnair_adata.obs["cell"] == "COP") | 
    (macnair_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, series_name + "_oligo_raw.h5ad"))

#### Validation Dataset

In [1]:
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

series_name = "macnair_validation"
clustered_h5ad_dir = find_env_dir("CLUSTERED_H5AD")
final_h5ad_dir = find_env_dir("FINAL_H5AD")

macnair_validation_adata = sc.read_h5ad(os.path.join(clustered_h5ad_dir, series_name + ".h5ad"))

In [ ]:
plot.plot_umap(macnair_validation_adata, series_name, has_celltype=True)

In [21]:
cell_marker_genes = {
    "OPC": ["PDGFRA", "MEGF11", "HAS2", "FERMT1", "CACNG5", "COL9A1", "MYT1", ],
    "COP": ["BCAN", "GPR17", "TCF7L2", "BCAS1", ],
    "Oligodendrocyte": ["MOG", "MAG", "CNP", "MOBP", "CLDN11",  
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8", "MYRF", ],
    "Neuron": ["STMN2", "CACNG3", "RBFOX3", "SNAP25", "ENO2", "SYT1", 
               "CCK", "GABRA1", "GRIN1", "GRIN2B", "MYT1L", "SYN1", "TMEM130", "CAMK2A", ],
    "Excitatory": ["SLC17A7", "SLC17A6", "NEUROD2", "DRD1", ],
    "Inhibitory": ["GAD1", "GAD2", "DLX5", "SLC32A1", ],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "ITGAM", "ITGAX", "SLC2A5", "TREM2", "HLA-DRA", "APOC1", ],
    "Tcell": ["CCL5", "CD3D", "CD3E", "CD8A", "NKG7", "TRAC", ],
    "Bcell": ["CD79A", "BANK1", "MS4A1", "XBP1", "SDC1", "MZB1", "TNFRSF17", ], 
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1", ],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3", ],
    "Astrocyte": ["GFAP", "ALDH1L1", "MLC1", "GJA1", "SLC14A1", "AGT", "BMPR1B", "ID4", "RFX4", "SOX9", "AQP4", ],
    "Ependymal": ["DNAH11", "FOXJ1", "DYNLRB2", "CIMAP3", ],
    "VLMC": ["COL1A2", "COL3A1", "DCN", ],
}
plot.plot_dotplot(macnair_validation_adata, series_name, cell_marker_genes, group = "leiden", project="ANNOTATION")

In [2]:
leiden_to_celltype = {
    "0": "Doublet",
    "1": "Astrocyte",
    "2": "Oligodendrocyte",
    "3": "Oligodendrocyte",
    "4": "Oligodendrocyte",
    "5": "Oligodendrocyte",
    "6": "ExcitatoryNeuron",
    "7": "Oligodendrocyte",
    "8": "Microglia",
    "9": "Oligodendrocyte",
    "10": "ExcitatoryNeuron",
    "11": "Oligodendrocyte",
    "12": "ExcitatoryNeuron",
    "13": "InhibitoryNeuron",
    "14": "Astrocyte",
    "15": "OPC",
    "16": "Oligodendrocyte",
    "17": "Oligodendrocyte",
    "18": "Astrocyte",
    "19": "Tcell",
    "20": "Oligodendrocyte",
    "21": "Endothelial",
    "22": "Bcell",
    "23": "Oligodendrocyte",
    "24": "Oligodendrocyte",
    "25": "VLMC",
    "26": "Oligodendrocyte",
    "27": "Ependymal",
    "28": "ExcitatoryNeuron",
    "29": "ExcitatoryNeuron",
    "30": "Oligodendrocyte",
    "31": "Oligodendrocyte",
    "32": "Doublet",
    "33": "Pericyte",
    "34": "Astrocyte",
    "35": "ExcitatoryNeuron",
    "36": "Oligodendrocyte",
    "37": "ExcitatoryNeuron",
    "38": "ExcitatoryNeuron",
    "39": "ExcitatoryNeuron",
    "40": "Microglia",
    "41": "InhibitoryNeuron",
    "42": "ExcitatoryNeuron",
    "43": "Microglia",
    "44": "ExcitatoryNeuron",
    "45": "InhibitoryNeuron",
    "46": "Microglia",
    "47": "ExcitatoryNeuron",
    "48": "InhibitoryNeuron",
    "49": "Astrocyte",
    "50": "Oligodendrocyte",
    "51": "Oligodendrocyte",
    "52": "ExcitatoryNeuron",
    "53": "InhibitoryNeuron",
    "54": "Microglia",
    "55": "Astrocyte",
    "56": "Oligodendrocyte",
    "57": "ExcitatoryNeuron",
    "58": "OPC",
    "59": "Microglia",
    "60": "ExcitatoryNeuron",
    "61": "ExcitatoryNeuron",
    "62": "ExcitatoryNeuron",
    "63": "COP", 
    "64": "ExcitatoryNeuron",
    "65": "Astrocyte",
}

leiden_str = macnair_validation_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
macnair_validation_adata.obs["cell"] = mapped

macnair_validation_adata = macnair_validation_adata[macnair_validation_adata.obs["cell"] != "Doublet"].copy()

In [3]:
macnair_validation_adata.write_h5ad(os.path.join(final_h5ad_dir, series_name + "_final.h5ad"))

In [4]:
pre_h5ad_dir = find_env_dir("PRE_H5AD")

opc = macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "OPC") |
    (macnair_validation_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(pre_h5ad_dir, series_name + "_opc_raw.h5ad"))

oligodendrocyte = macnair_validation_adata[macnair_validation_adata.obs["cell"] == "Oligodendrocyte"]
oligodendrocyte.write_h5ad(os.path.join(pre_h5ad_dir, series_name + "_oligodendrocyte_raw.h5ad"))

oligo= macnair_validation_adata[
    (macnair_validation_adata.obs["cell"] == "OPC") |
    (macnair_validation_adata.obs["cell"] == "COP") | 
    (macnair_validation_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(pre_h5ad_dir, series_name + "_oligo_raw.h5ad"))